# 02. Predictive Delinquency Modeling & Explainability
**Client:** Geldium Finance | **Focus:** Regulated Credit Risk & Explainable AI

### Objectives:
1. Train an interpretable classifier (Logistic Regression with balanced weighting).
2. Evaluate model performance via ROC-AUC, Precision, Recall, and Confusion Matrix.
3. Extract feature importance coefficients for fair lending compliance.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, confusion_matrix

In [2]:
# Prepare train-test split
df = pd.read_csv('../data/delinquency_prediction_dataset.csv')
features = ['age', 'annual_income', 'credit_limit', 'current_balance', 
            'credit_utilization_ratio', 'debt_to_income_ratio', 'missed_payments_6m']
X = df[features]
y = df['delinquent_flag']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [3]:
# Train Logistic Regression model
model = LogisticRegression(class_weight='balanced', random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print('Classification Report:\n', classification_report(y_test, y_pred))
print(f'AUC-ROC Score: {roc_auc_score(y_test, y_prob):.4f}')

In [4]:
# Explainability: Standardized Feature Coefficients
coef_df = pd.DataFrame({
    'Feature': features,
    'Coefficient': model.coef_[0]
}).sort_values(by='Coefficient', ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(x='Coefficient', y='Feature', data=coef_df, palette='viridis')
plt.title('Standardized Feature Coefficients (Impact on Delinquency Odds)')
plt.tight_layout()
plt.show()

### Summary & Compliance Note:
- **Transparency:** Linear coefficients allow straightforward explanation of adverse actions to regulators.
- **Key Risk Drivers:** Credit utilization ratio and recent missed payments provide the highest positive weight toward delinquency predictions.